# Facial Emotion CNN — Kaggle GPU training

Trains the CNN defined in `src/facial_emotion/model/cnn.py` on FER2013
(`msambare/fer2013`) using CLAHE + grayscale + normalization preprocessing
and inline augmentation. Run this notebook as a Kaggle Notebook with the
`msambare/fer2013` dataset attached and GPU (T4) enabled.

Outputs (download these into the repo's `artifacts/` folder afterward):
- `model.keras` — trained model weights
- `history.json` — per-epoch train/val loss & accuracy
- `metrics.json` — held-out PrivateTest accuracy/precision/recall/F1
- `confusion_matrix.png`

In [ ]:
# On Kaggle: attach the msambare/fer2013 dataset via 'Add Input', then
# clone/copy this repo's src/ so imports below work, e.g.:
#   !git clone https://github.com/<user>/facial-emotion-cnn.git repo
#   import sys; sys.path.insert(0, 'repo/src')
import json
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('repo/src')))  # adjust if the layout differs

from facial_emotion.constants import EMOTION_LABELS, IMG_SIZE
from facial_emotion.data.dataset import build_datasets
from facial_emotion.model.cnn import build_cnn, compile_model

print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
DATA_DIR = Path('/kaggle/input/fer2013')  # msambare/fer2013 layout: train/, test/
OUT_DIR = Path('/kaggle/working/artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 60

train_ds, val_ds, test_ds, class_names = build_datasets(
    DATA_DIR, img_size=IMG_SIZE, batch_size=BATCH_SIZE, val_split=0.1, use_clahe=True
)
print('Classes:', class_names)

In [ ]:
model = compile_model(build_cnn(use_augmentation=True))
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

with open(OUT_DIR / 'history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

In [ ]:
model.save(OUT_DIR / 'model.keras')

In [ ]:
# Real held-out evaluation on the FER2013 test split (PrivateTest-equivalent)
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(labels.numpy().tolist())

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
cm = confusion_matrix(y_true, y_pred)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

metrics = {
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
    'epochs_trained': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('FER2013 test confusion matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'training_curves.png', dpi=150)
plt.show()